In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the feature-engineered dataset
print("Loading feature-engineered dataset...")
df = pd.read_csv('/workspace/Renewable-Energy-Prediction/data/processed/Feature_Engineering _Dataset/feature_engineered_data.csv', 
                 parse_dates=['time'])
print(f"Dataset shape: {df.shape}")


Loading feature-engineered dataset...


Dataset shape: (177210, 44)


In [4]:
# Check for missing values from the lag features
print("\nMissing values:")
print(df.isnull().sum().sum())


Missing values:
1464


In [5]:
# Fill missing values for lag features
print("\nFilling missing values...")
df = df.fillna(method='bfill')
missing_after = df.isnull().sum().sum()
print(f"Remaining missing values: {missing_after}")



Filling missing values...
Remaining missing values: 0


In [6]:
# Define feature columns
feature_cols = [col for col in df.columns if col not in ['Area', 'YEAR', 'Country', 'time', 'Solar', 'Wind Onshore']]
print(f"\nNumber of features: {len(feature_cols)}")


Number of features: 38


In [7]:
# Define evaluation function
def evaluate_model(y_true, y_pred, model_name, target_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"{model_name} - {target_name} Metrics:")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R²: {r2:.4f}")
    
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

In [8]:
# Model training function for a specific target (Wind and Solar)
def train_evaluate_models(X_train, X_test, y_train, y_test, target_name):
    results = {}
    models = {}
    
    print(f"\n===== Training models for {target_name} prediction =====")
    
    # 1. Random Forest
    print("\nTraining Random Forest...")
    start_time = time.time()
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    rf_time = time.time() - start_time
    print(f"Training time: {rf_time:.2f} seconds")
    
    rf_results = evaluate_model(y_test, rf_pred, "Random Forest", target_name)
    results["Random Forest"] = rf_results
    models["Random Forest"] = rf_model
    
    # Feature importance for Random Forest
    rf_feature_imp = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False).head(10)
    
    print("\nTop 10 features for Random Forest:")
    print(rf_feature_imp)
    
    # 2. Gradient Boosting Machine
    print("\nTraining GBM...")
    start_time = time.time()
    gbm_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    gbm_model.fit(X_train, y_train)
    gbm_pred = gbm_model.predict(X_test)
    gbm_time = time.time() - start_time
    print(f"Training time: {gbm_time:.2f} seconds")
    
    gbm_results = evaluate_model(y_test, gbm_pred, "GBM", target_name)
    results["GBM"] = gbm_results
    models["GBM"] = gbm_model
    
    # 3. XGBoost
    print("\nTraining XGBoost...")
    start_time = time.time()
    xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_test)
    xgb_time = time.time() - start_time
    print(f"Training time: {xgb_time:.2f} seconds")
    
    xgb_results = evaluate_model(y_test, xgb_pred, "XGBoost", target_name)
    results["XGBoost"] = xgb_results
    models["XGBoost"] = xgb_model
    
    # 4. Simple Ensemble (Voting Regressor)
    print("\nTraining Simple Ensemble (Voting Regressor)...")
    start_time = time.time()
    ensemble_model = VotingRegressor(
        estimators=[
            ('rf', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
            ('gbm', GradientBoostingRegressor(n_estimators=100, random_state=42)),
            ('xgb', xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1))
        ]
    )
    ensemble_model.fit(X_train, y_train)
    ensemble_pred = ensemble_model.predict(X_test)
    ensemble_time = time.time() - start_time
    print(f"Training time: {ensemble_time:.2f} seconds")
    
    ensemble_results = evaluate_model(y_test, ensemble_pred, "Ensemble", target_name)
    results["Ensemble"] = ensemble_results
    models["Ensemble"] = ensemble_model
    
    # 5. Plot actual vs predicted
    plt.figure(figsize=(12, 6))
    plt.plot(y_test[:100], label='Actual', color='blue', alpha=0.7)
    plt.plot(rf_pred[:100], label='Random Forest', color='green', alpha=0.7)
    plt.plot(gbm_pred[:100], label='GBM', color='red', alpha=0.7)
    plt.plot(xgb_pred[:100], label='XGBoost', color='purple', alpha=0.7)
    plt.plot(ensemble_pred[:100], label='Ensemble', color='orange', alpha=0.7)
    plt.xlabel('Sample Index')
    plt.ylabel(f'{target_name} Generation')
    plt.title(f'Actual vs Predicted {target_name} Generation (First 100 samples)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'/workspace/Renewable-Energy-Prediction/results/{target_name.lower()}_predictions_comparison.png')
    
    # Compare model performance
    performance_df = pd.DataFrame(results).T
    print("\nModel Performance Comparison:")
    print(performance_df)
    
    # Save models
    for model_name, model in models.items():
        joblib.dump(model, f'/workspace/Renewable-Energy-Prediction/models/{target_name.lower()}_{model_name.lower().replace(" ", "_")}_model.pkl')
    
    return results, models


In [9]:
# Main execution
print("\nPreparing data for modeling...")


Preparing data for modeling...


In [10]:
# Time-based split (more appropriate for time-series data)
# We'll use the last 20% of the data as test set to maintain time ordering
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print(f"Training set size: {train_df.shape}")
print(f"Test set size: {test_df.shape}")

Training set size: (141768, 44)
Test set size: (35442, 44)


In [11]:
# Prepare features and targets
X_train = train_df[feature_cols]
y_train_solar = train_df['Solar']
y_train_wind = train_df['Wind Onshore']

X_test = test_df[feature_cols]
y_test_solar = test_df['Solar']
y_test_wind = test_df['Wind Onshore']

In [12]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [13]:
# Convert back to DataFrame to maintain column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [14]:
# Save the scaler
joblib.dump(scaler, '/workspace/Renewable-Energy-Prediction/models/feature_scaler.pkl')

['/workspace/Renewable-Energy-Prediction/models/feature_scaler.pkl']